# Links

## Imports

In [1]:
import pandas as pd

In [2]:
root = '../data'

## Read file

In [3]:
# read every column as strings first to inspect the raw format
links_raw = pd.read_csv(root + '/links.csv', dtype=str)
print(f"There are {links_raw.shape[0]} rows and {links_raw.shape[1]} columns in the links dataset")
display(links_raw.head())

# check that all values are digits and count the number of digits per column
digits = r'\d+'
for column_name in links_raw.columns:
    c = links_raw[column_name].dropna()
    print(f"{column_name}: {(~c.str.fullmatch(digits)).sum()} values that are not digits, number of digits: {c.str.len().value_counts().sort_index().to_dict()}")

There are 45843 rows and 3 columns in the links dataset


,movieId,imdbId,tmdbId
0,1,0114709,862
1,2,0113497,8844
2,3,0113228,15602
3,4,0114885,31357
4,5,0113041,11862


movieId: 0 values that are not digits, number of digits: {1: 9, 2: 89, 3: 883, 4: 7356, 5: 12128, 6: 25378}
imdbId: 0 values that are not digits, number of digits: {7: 45843}
tmdbId: 0 values that are not digits, number of digits: {1: 4, 2: 59, 3: 733, 4: 2992, 5: 25662, 6: 16174}


In [4]:
# imdbId always has 7 digits with leading zeros, which are dropped when read as a number
# tmdbId has missing values and would be read as float, so it is read as a nullable integer
links = pd.read_csv(root + '/links.csv', dtype={'tmdbId': 'Int64'})
display(links.dtypes)

print(f"Number of duplicate rows: {links.duplicated().sum()}")

movieId    int64
imdbId     int64
tmdbId     Int64
dtype: object

Number of duplicate rows: 0


## movieId

In [ ]:
# movieId is the MovieLens id, used in ratings
print(f"Nr of rows with no movieId: {links.movieId.isna().sum()}")
print(f"Nr of duplicate movieIds: {links.movieId.duplicated().sum()}")

Nr of rows with no movieId: 0
Nr of duplicate movieIds: 0


count     45843.000000
mean      96578.775626
std       57216.863469
min           1.000000
25%       49202.500000
50%      108799.000000
75%      145270.500000
max      176279.000000
Name: movieId, dtype: float64

## imdbId

In [6]:
print(f"Nr of rows with no imdbId: {links.imdbId.isna().sum()}")
print(f"Nr of duplicate imdbIds: {links.imdbId.duplicated().sum()}")

# imdb_id in movies_metadata is written as tt + 7 digits, e.g. links 0114709 = movies_metadata tt0114709
print(f"imdbId of the first movie in links: {links_raw.imdbId.iloc[0]}")

Nr of rows with no imdbId: 0
Nr of duplicate imdbIds: 0
imdbId of the first movie in links: 0114709


## tmdbId

In [7]:
print(f"There are {links.tmdbId.isna().sum()} ({links.tmdbId.isna().mean():.1%}) rows with no tmdbId")
display(links[links.tmdbId.isna()].head())

# check whether tmdbId is unique
tmdb_vc = links.tmdbId.value_counts()
print(f"There are {links.tmdbId.dropna().duplicated().sum()} duplicate rows, among which the highest number of duplicates for one id is {tmdb_vc.iloc[0]} which occurs {(tmdb_vc == tmdb_vc.iloc[0]).sum()} time")

# the duplicate tmdbIds belong to different movieIds and imdbIds
dup_tmdb = links.tmdbId[links.tmdbId.duplicated(keep=False)].dropna().unique()
display(links[links.tmdbId.isin(dup_tmdb)].sort_values(['tmdbId', 'movieId']).head(6))

There are 219 (0.5%) rows with no tmdbId


,movieId,imdbId,tmdbId
140,142,94878,<NA>
598,604,115978,<NA>
708,720,118114,<NA>
709,721,114103,<NA>
718,730,125877,<NA>


There are 30 duplicate rows, among which the highest number of duplicates for one id is 3 which occurs 1 time


,movieId,imdbId,tmdbId
5905,6003,290538,4912
34144,144606,270288,4912
7388,7587,62229,5511
9215,27136,165303,5511
4148,4241,266860,10991
45197,174533,235679,10991


## movies_metadata

In [8]:
# read the ids from movies_metadata, ids that cannot be read as numbers come from the wrongfully split rows
mm = pd.read_csv(root + '/movies_metadata.csv', usecols=['id', 'imdb_id', 'title', 'release_date'], dtype=str)
mm['id'] = pd.to_numeric(mm.id, errors='coerce')
mm = mm.dropna(subset=['id']).astype({'id': int}).drop_duplicates('id')
mm['imdbId'] = pd.to_numeric(mm.imdb_id.str[2:], errors='coerce') # remove tt to compare with links

print(f"Nr of movies in movies_metadata that are not in links: {(~mm.id.isin(links.tmdbId)).sum()}")

# tmdbIds in links that are not in movies_metadata, and whether they can be found through imdbId instead
not_in_mm = links[links.tmdbId.notna() & ~links.tmdbId.isin(mm.id)]
print(f"Nr of tmdbIds in links that are not in movies_metadata: {len(not_in_mm)}, of which {not_in_mm.imdbId.isin(mm.imdbId).sum()} can be found by imdbId")
print(f"Nr of rows without tmdbId that can be found by imdbId: {links[links.tmdbId.isna()].imdbId.isin(mm.imdbId).sum()}")

Nr of movies in movies_metadata that are not in links: 0
Nr of tmdbIds in links that are not in movies_metadata: 161, of which 0 can be found by imdbId
Nr of rows without tmdbId that can be found by imdbId: 0


In [9]:
# compare imdbId for the movies that match on tmdbId
joined = links.merge(mm, left_on='tmdbId', right_on='id', suffixes=('', '_mm'))
mismatch = joined[joined.imdbId != joined.imdbId_mm]
print(f"Nr of movies where imdbId differs from movies_metadata: {len(mismatch)}, of which {mismatch.imdbId_mm.isna().sum()} have no imdb_id in movies_metadata")

mismatch = mismatch.dropna(subset=['imdbId_mm'])
print(f"Nr of real mismatches: {len(mismatch)}, of which {mismatch.tmdbId.isin(dup_tmdb).sum()} have a duplicate tmdbId")

# for the duplicate tmdbIds, count how many of the rows have the same imdbId as movies_metadata
matches = joined[joined.tmdbId.isin(dup_tmdb)].assign(match=lambda d: d.imdbId == d.imdbId_mm).groupby('tmdbId')['match'].sum()
print("Nr of rows per duplicate tmdbId that match movies_metadata:")
display(matches.value_counts())

# the remaining mismatches, the tmdbId points to a movie with the same title but another imdbId (e.g. another release year)
display(mismatch[~mismatch.tmdbId.isin(dup_tmdb)][['movieId', 'imdbId', 'tmdbId', 'imdb_id', 'title', 'release_date']].head())

Nr of movies where imdbId differs from movies_metadata: 112, of which 17 have no imdb_id in movies_metadata
Nr of real mismatches: 95, of which 30 have a duplicate tmdbId
Nr of rows per duplicate tmdbId that match movies_metadata:


match
1    29
Name: count, dtype: int64

,movieId,imdbId,tmdbId,imdb_id,title,release_date
707,723,92123,131232,tt0333373,Two Friends,2002-03-20
868,895,105729,79782,tt1684935,Venice,2010-05-25
1081,1115,114472,141210,tt2250194,The Sleepover,2013-10-12
2114,2223,18876,143750,tt2140519,The Farmer's Wife,2012-06-20
2397,2510,134948,26425,tt0123221,Just the Ticket,1999-02-26


## credits

In [10]:
credit_ids = pd.read_csv(root + '/credits.csv', usecols=['id']).id
print(f"Nr of tmdbIds in links that are not in credits: {(links.tmdbId.notna() & ~links.tmdbId.isin(credit_ids)).sum()}")
print(f"Nr of movies in credits that are not in links: {(~credit_ids.drop_duplicates().isin(links.tmdbId)).sum()}")

Nr of tmdbIds in links that are not in credits: 162
Nr of movies in credits that are not in links: 0


## ratings

In [11]:
# only movieId is needed, the file has 26 million rows
ratings_movies = pd.read_csv(root + '/ratings.csv', usecols=['movieId']).movieId
rated = ratings_movies.drop_duplicates()
print(f"There are {len(rated)} rated movies, of which {(~rated.isin(links.movieId)).sum()} are not in links")
print(f"Nr of movies in links without ratings: {(~links.movieId.isin(rated)).sum()}")

# ratings for movies that cannot be joined with movies_metadata (no tmdbId, or tmdbId not in movies_metadata)
no_metadata = links.loc[~links.tmdbId.isin(mm.id).fillna(False), 'movieId']
print(f"Nr of movies that cannot be joined with movies_metadata: {len(no_metadata)}")
print(f"Nr of ratings for these movies: {ratings_movies.isin(no_metadata).sum()} ({ratings_movies.isin(no_metadata).mean():.2%})")

There are 45115 rated movies, of which 0 are not in links
Nr of movies in links without ratings: 728
Nr of movies that cannot be joined with movies_metadata: 380
Nr of ratings for these movies: 42707 (0.16%)


## links_small

In [12]:
links_small = pd.read_csv(root + '/links_small.csv', dtype={'tmdbId': 'Int64'})
print(f"There are {links_small.shape[0]} rows and {links_small.shape[1]} columns in the links_small dataset")
print(f"Number of duplicate rows: {links_small.duplicated().sum()}")

for column_name in links_small.columns:
    c = links_small[column_name]
    print(f"{column_name}: {c.isna().sum()} missing, {c.dropna().duplicated().sum()} duplicates")

There are 9125 rows and 3 columns in the links_small dataset
Number of duplicate rows: 0
movieId: 0 missing, 0 duplicates
imdbId: 0 missing, 0 duplicates
tmdbId: 13 missing, 0 duplicates


In [13]:
# check whether links_small is a subset of links
not_in_links = links_small[~links_small.movieId.isin(links.movieId)]
print(f"Nr of movieIds in links_small that are not in links: {len(not_in_links)}")
display(not_in_links)

# two of them exist in links with the same imdbId, but another movieId and tmdbId
display(not_in_links.merge(links, on='imdbId', suffixes=('_small', '')))

# the movies in both files have the same ids
common = links_small.merge(links, on='movieId', suffixes=('_small', ''))
print(f"Nr of common movies with different imdbId: {(common.imdbId_small != common.imdbId).sum()}")
print(f"Nr of common movies with different tmdbId: {(common.tmdbId_small.fillna(-1) != common.tmdbId.fillna(-1)).sum()}")

Nr of movieIds in links_small that are not in links: 10


,movieId,imdbId,tmdbId
5173,7502,185906,191903
7527,77658,81846,205300
8045,94466,2085059,<NA>
8505,108548,898266,<NA>
8507,108583,72500,<NA>
8520,108979,213338,314040
9062,150856,5189670,374454
9118,162376,4574334,410612
9123,164977,27660,137608
9124,164979,3447228,410803


,movieId_small,imdbId,tmdbId_small,movieId,tmdbId
0,7502,185906,191903,170705,331214
1,77658,81846,205300,171495,409926


Nr of common movies with different imdbId: 0
Nr of common movies with different tmdbId: 0


In [14]:
# ratings_small should only contain movies from links_small
ratings_small_movies = pd.read_csv(root + '/ratings_small.csv', usecols=['movieId']).movieId.drop_duplicates()
print(f"There are {len(ratings_small_movies)} rated movies in ratings_small, of which {(~ratings_small_movies.isin(links_small.movieId)).sum()} are not in links_small")
print(f"Nr of movies in links_small without ratings in ratings_small: {(~links_small.movieId.isin(ratings_small_movies)).sum()}")

There are 9066 rated movies in ratings_small, of which 0 are not in links_small
Nr of movies in links_small without ratings in ratings_small: 59


## Conclusion

**Reading the file**

- `links.csv` loads cleanly with **45,843 rows and 3 columns** (`movieId`, `imdbId`, `tmdbId`), and every value is a whole number.
- `imdbId` is always stored as 7 digits with leading zeros, which are dropped when read as a number. To join with `imdb_id` in movies_metadata (written as `tt` + 7 digits) the format must be converted.
- `tmdbId` has missing values and is read as float by default, so it is read as a nullable integer (`Int64`).

**Duplicates and keys**
- `movieId` (the MovieLens id used in ratings) and `imdbId` are unique and have no missing values.
- `tmdbId` (the id in movies_metadata and credits) is **not unique**: 29 tmdbIds occur more than once (30 surplus rows, one id three times), each time with a different `movieId` and `imdbId`. In every case exactly one of the rows has the same imdbId as movies_metadata, so the other rows link to the wrong movie.
- Apart from the duplicates, 65 more rows have a tmdbId whose imdbId in movies_metadata differs from the one in links. They appear to point to another movie with the same title, e.g. *Two Friends* links to a movie from 2002. These links are unreliable. A further 17 cannot be checked because movies_metadata has no `imdb_id`.

**Missing values**

| Column | Encoded as | Rows |
|---|---|---|
| `tmdbId` (links) | `NaN` | 219 (0.5 %) |
| `tmdbId` (links_small) | `NaN` | 13 (0.1 %) |

The other columns have no missing values.

**Joins with the other files**
- Every movie in movies_metadata and credits has a link, but 161 tmdbIds in links (162 for credits) do not exist in movies_metadata. They cannot be found by imdbId either. Together with the missing tmdbIds, **380 movies (0.8 %) cannot be joined with movies_metadata**. They have 42,707 ratings (0.16 %).
- All 45,115 rated movies in ratings.csv are in links, while 728 movies in links (1.6 %) have no ratings.

**links_small**
- `links_small.csv` has **9,125 rows** and no duplicates in any column.
- It is **not a strict subset of links**: 10 movieIds do not exist in links. Two of them are in links with the same imdbId but another movieId and tmdbId, so the ids have changed between versions of the dataset. The movies that are in both files have the same ids.
- All 9,066 movies in ratings_small are in links_small, while 59 movies in links_small (0.6 %) have no ratings in ratings_small.